In [ ]:
# mount google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# clone github repo
%cd /content
!rm -rf aot-cpl-point-bot
!git clone https://github.com/mduboef/aot-cpl-point-bot.git
%cd /content/aot-cpl-point-bot
!pip install -q -r requirements.txt

/content
Cloning into 'aot-cpl-point-bot'...
remote: Enumerating objects: 1627, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 1627 (delta 14), reused 19 (delta 8), pack-reused 1597 (from 1)
Receiving objects: 100% (1627/1627), 36.79 MiB | 37.55 MiB/s, done.
Resolving deltas: 100% (520/520), done.
/content/aot-cpl-point-bot
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 30.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [ ]:
# sanity check
import os, torch, pickle, subprocess
print("CUDA:", torch.cuda.is_available())

# preference datasets to train + evaluate on (number = GREY_WEIGHT used by preferenceGen.py)
rewardWeights = [175, 180, 185, 190, 195]
for i in rewardWeights:
    # check for both training and testing set
    for p in [f"data/trainingData/trainPreferences{i}.pkl", f"data/testingData/testPreferences{i}.pkl"]:
        with open(p, "rb") as f:
            print(p, "→", len(pickle.load(f)), "pairs")

# one BC reference policy per dataset, stored in models/REF_POLICY_{i}
# if one isn't in the GH models directory, train it now
for i in rewardWeights:
    refPath = f"models/REF_POLICY_{i}/policy.pt"
    print(f"REF_POLICY_{i} cached:", os.path.isfile(refPath))
    if not os.path.isfile(refPath):
        print(f"No reference policy for dataset {i}. Training one now.")
        subprocess.run(["python3", "train.py", "--method", "ref", "-data", str(i)], check=True)


CUDA: False
data/trainingData/trainPreferences.pkl → 13041 pairs
data/testingData/testPreferences.pkl → 13041 pairs
REF_POLICY cached: True


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir models

# view TensorBoard logs saved to Drive (only the drive.mount cell needs to run first)
tbStamp = "20261010_001553"   # set tbStamp to a past run's timestamp folder, or None to use the local models/ dir from this session
tbDir   = "models" if tbStamp is None else \
          f"/content/drive/MyDrive/UMass/CPL (Independent Study Fall 2026)/Distributional CPL/runs/{tbStamp}/models"
%load_ext tensorboard
%tensorboard --logdir "$tbDir"

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 26524), started 0:00:21 ago. (Use '!kill 26524' to kill it.)

<IPython.core.display.Javascript object>

In [ ]:
# train every (method, bias, dataset) combination, save each run to Drive when it finishes
import os, re, shutil, datetime, time, subprocess

# train method once for every listed bias value
methods   = {
    "bc"            : [],   # empty vec -> no -bias flag
    "cpl"           : [0.75, 1.0],
    "cpl_paot"      : [0.75],
    "cpl_uaot"      : [0.75],
    "cpl_uaot_ref"  : [0.75],
}

# TEMP (step 6): every (method, bias) is trained once per preference dataset in rewardWeights (set in the sanity check cell)
# run dirs are named models/<METHOD>_<bias>_<dataset>_N (or models/<METHOD>_<dataset>_N for bc)

# flatten into one job per run: (method, bias, dataset), bias=None means no -bias flag
jobs = [(m, None if b is None else float(b), i)
        for i in rewardWeights for m, biases in methods.items() for b in (biases or [None])]

stamp     = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # timestamp used to differentate runs in drive
driveRoot = f"/content/drive/MyDrive/UMass/CPL (Independent Study Fall 2026)/Distributional CPL/runs/{stamp}"
logDir    = os.path.join(driveRoot, "logs")
os.makedirs(logDir, exist_ok=True)

# save the per-dataset reference policies alongside the runs that use them
for i in rewardWeights:
    shutil.copytree(f"models/REF_POLICY_{i}", os.path.join(driveRoot, "models", f"REF_POLICY_{i}"), dirs_exist_ok=True)

# cap concurrent runs so the procs don't oversubscribe the CPU cores (or RAM); extra jobs queue
nCores      = os.cpu_count() or 8
maxParallel = min(len(jobs), nCores)
perProc     = max(1, nCores // maxParallel)
env         = dict(os.environ, OMP_NUM_THREADS=str(perProc), MKL_NUM_THREADS=str(perProc))

# label used for log files / printing, and the run dir tag train.py will create:
# <METHOD>_<bias>_<dataset>_N for contrastive methods, <METHOD>_<dataset>_N for bc
def jobLabel(m, bias, i):
    return f"{m}_{i}" if bias is None else f"{m}_{bias}_{i}"
def jobTag(m, bias, i):
    return f"{m.upper()}_{i}" if bias is None else f"{m.upper()}_{bias}_{i}"

before = set(os.listdir("models"))

def launch(m, bias, i):
    lf  = open(os.path.join(logDir, f"{jobLabel(m, bias, i)}.log"), "w")
    cmd = ["python3", "-u", "train.py", "--method", m, "-data", str(i)]
    if bias is not None:
        cmd += ["-bias", str(bias)]
    p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, env=env)
    print(f"launched {jobLabel(m, bias, i):28s} pid {p.pid}  (OMP_NUM_THREADS={perProc})", flush=True)
    return p, lf

# launch up to maxParallel jobs at once; each streams its console output to its own log file in Drive
# as each finishes, copy its exact new run dir to Drive immediately and start the next queued job
pending = list(jobs)
running = {}   # (m, bias, dataset) -> (proc, logFile)
saved   = set()
while pending or running:
    while pending and len(running) < maxParallel:
        job = pending.pop(0)
        running[job] = launch(*job)
    time.sleep(15)
    for (m, bias, i), (p, lf) in list(running.items()):
        if p.poll() is None:
            continue
        lf.flush(); lf.close()
        label  = jobLabel(m, bias, i)
        status = "OK" if p.returncode == 0 else f"FAILED(rc={p.returncode})"
        # match ONLY this method + bias + dataset: ^TAG_<digits>$  (so 'cpl' won't grab 'cpl_paot', 0.1 won't grab 0.15, etc.)
        pat = re.compile(rf"^{re.escape(jobTag(m, bias, i))}_\d+$")
        new = [d for d in os.listdir("models")
                if d not in before and d not in saved and pat.match(d)]
        for d in new:
            shutil.copytree(os.path.join("models", d),
                            os.path.join(driveRoot, "models", d), dirs_exist_ok=True)
            saved.add(d)
            print(f"[{label}] {status} → saved models/{d} to Drive", flush=True)
        if not new:
            print(f"[{label}] {status} but no new run dir found — check logs/{label}.log", flush=True)
        del running[(m, bias, i)]

print("\nALL DONE →", driveRoot)

launched bc                       pid 3070  (OMP_NUM_THREADS=1)
launched cpl_0.1                  pid 3071  (OMP_NUM_THREADS=1)
launched cpl_0.25                 pid 3072  (OMP_NUM_THREADS=1)
launched cpl_0.5                  pid 3073  (OMP_NUM_THREADS=1)
launched cpl_0.75                 pid 3074  (OMP_NUM_THREADS=1)
launched cpl_1.0                  pid 3075  (OMP_NUM_THREADS=1)
launched cpl_paot_0.1             pid 3076  (OMP_NUM_THREADS=1)
launched cpl_paot_0.25            pid 3077  (OMP_NUM_THREADS=1)
[bc] OK → saved models/BC_1 to Drive
launched cpl_paot_0.5             pid 6892  (OMP_NUM_THREADS=1)
[cpl_paot_0.1] OK → saved models/CPL_PAOT_0.1_1 to Drive
[cpl_paot_0.25] OK → saved models/CPL_PAOT_0.25_1 to Drive
launched cpl_paot_0.75            pid 16274  (OMP_NUM_THREADS=1)
launched cpl_paot_1.0             pid 16275  (OMP_NUM_THREADS=1)
[cpl_paot_0.5] OK → saved models/CPL_PAOT_0.5_1 to Drive
launched cpl_uaot_0.1             pid 20014  (OMP_NUM_THREADS=1)
[cpl_0.5] OK → save

In [ ]:
# print final eval metrics for newly trained models, one table per preference dataset
import os, re, json

# must match the methods dict in the training cell
methods   = {
    "bc"            : [],
    "cpl"           : [0.75, 1.0],
    "cpl_paot"      : [0.75],
    "cpl_uaot"      : [0.75],
    "cpl_uaot_ref"  : [0.75],
}

split   = "test"  # held-out set; switch to "train" to see fit on training pairs

# for each (method, bias, dataset) pick the run dir created this session (highest N for the exact tag)
def latestRunDir(method, bias, i):
    tag = method.upper() if bias is None else f"{method.upper()}_{float(bias)}"
    tag += f"_{i}"
    pat = re.compile(rf"^{re.escape(tag)}_(\d+)$")
    if not os.path.exists("models"):
        return None
    matches = [
        (int(pat.match(d).group(1)), d)
        for d in os.listdir("models") if pat.match(d)
    ]
    return os.path.join("models", max(matches)[1]) if matches else None

header = ["Method", "λ", "Orig-Pair Acc", "pAOT Loss", "pAOT Viol Rate", "uAOT Loss", "uAOT Viol Rate", "Avg Reward"]

# TEMP (step 6): one table per dataset in rewardWeights (set in the sanity check cell)
for i in rewardWeights:
    rows = []
    for m, biases in methods.items():
        for bias in (biases or [None]):
            biasStr = "-" if bias is None else str(float(bias))
            runDir  = latestRunDir(m, bias, i)
            if runDir is None:
                rows.append([m, biasStr, "no run dir", "", "", "", "", ""])
                continue
            with open(os.path.join(runDir, "eval_stats.json")) as f:
                stats = json.load(f)
            acc   = stats["preferenceAccuracy"][split]["overall"]
            paot  = stats["paotLoss"][split]
            uaot  = stats["uaotLoss"][split]
            avgReward = sum(r["reward"] for r in stats["rollouts"]) / len(stats["rollouts"])
            rows.append([
                m,
                biasStr,
                f"{acc:.1%}",
                f"{paot['paotLoss']:.4f}",
                f"{paot['violationFreq']:.1%}",
                f"{uaot['uaotLoss']:.4f}",
                f"{uaot['violationFreq']:.1%}",
                f"{avgReward:.1f}",
            ])

    # print table of eval metrics
    widths = [max(len(str(r[c])) for r in ([header] + rows)) for c in range(len(header))]
    def fmtRow(r):
        return "  ".join(str(cell).ljust(widths[c]) for c, cell in enumerate(r))
    print(f"\n=== dataset {i}: evaluation metrics on the {split} set ===\n")
    print(fmtRow(header))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print(fmtRow(r))

evaluation metrics on the test set

Method        λ     Orig-Pair Acc  pAOT Loss  pAOT Viol Rate  uAOT Loss  uAOT Viol Rate  Avg Reward
------------  ----  -------------  ---------  --------------  ---------  --------------  ----------
bc            -     72.8%          0.9302     26.2%           1.0463     45.2%           -8569.9   
cpl           0.1   74.4%          1.2094     62.4%           1.1390     64.1%           -6704.8   
cpl           0.25  75.8%          0.9989     51.4%           1.0351     57.0%           -7310.3   
cpl           0.5   79.8%          0.3073     8.9%            0.4174     24.5%           -7098.6   
cpl           0.75  85.2%          0.1146     0.0%            0.2566     8.1%            -5868.2   
cpl           1.0   91.4%          0.0312     0.5%            0.0566     1.3%            -15308.1  
cpl_paot      0.1   73.8%          0.3372     12.9%           0.2532     1.9%            -8875.2   
cpl_paot      0.25  76.5%          0.1964     0.6%            0.